In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/tlp25_IRP/npy_files.zip -d /content/

In [ ]:
!pip install pandas==2.3.3 livelossplot torchinfo -q

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from torchinfo import summary
import random
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader
import torch.nn as nn
from livelossplot import PlotLosses
import torch.nn.functional as F

sys.path.append("/content/drive/MyDrive/tlp25_IRP/Modelling")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.0/626.0 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 140.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.16.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.


In [ ]:
sys.path.append("/content/drive/MyDrive/tlp25_IRP/Modelling")

from unet import MultiHeadUNET
from Plot_dataset import plot_dataset
from get_norm_stats import get_normalization_stats
from dataset import UHSDataset, transform_dataset

In [ ]:
def set_device(device="cpu", idx=0):
    if device != "cpu":
        if torch.cuda.device_count() > idx and torch.cuda.is_available():
            print("Cuda installed! Running on GPU {} {}!".format(idx, torch.cuda.get_device_name(idx)))
            device="cuda:{}".format(idx)
        elif torch.cuda.device_count() > 0 and torch.cuda.is_available():
            print("Cuda installed but only {} GPU(s) available! Running on GPU 0 {}!".format(torch.cuda.device_count(), torch.cuda.get_device_name()))
            device="cuda:0"
        else:
            device="cpu"
            print("No GPU available! Running on CPU")
    return device

device = set_device("cuda")

def set_seed(seed=42):
    """
    Set the random seed for reproducibility.

    Parameters:
    ----------
    seed (int): The seed value to set for random number generation.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True

    return True

Cuda installed! Running on GPU 0 NVIDIA A100-SXM4-40GB!


In [ ]:
npy_file_dir = "/content/content/drive/MyDrive/tlp25_IRP/npy_files/"

site_dirs = sorted(os.listdir(npy_file_dir))

train_files, val_test_files = train_test_split(site_dirs, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(val_test_files, test_size = 0.5, random_state=42)

In [ ]:
print(len(train_files))

730


In [ ]:
stats_dict = get_normalization_stats(train_dataset=train_files)

{'min_fcmip': 15634616.0, 'max_fcmip': 21247800.0, 'min_fcmpr': 0.0, 'max_fcmpr': 21179.091796875, 'min_fcmir': 0.0, 'max_fcmir': 25313.69921875, 'min_pressure': np.float32(96.28905), 'max_pressure': np.float32(104.03392)}


In [ ]:
transform = transform_dataset
dir = '/content/content/drive/MyDrive/tlp25_IRP/npy_files/'
train_dataset = UHSDataset(train_files, transform=transform, norm_stats=stats_dict, dir=dir)
val_dataset = UHSDataset(val_files, transform=transform, norm_stats=stats_dict, dir=dir)
test_dataset = UHSDataset(test_files, transform=transform, norm_stats=stats_dict, dir=dir)

In [ ]:
model = MultiHeadUNET(in_channels=2, grid_out_channels=2).to(device)
grid_input = torch.ones(1, 2, 12, 61, 61).to(device)
cond_input = torch.randn(1, 7).to(device)
spatial, injrate, withrate, moles_inplace = model(grid_input, cond_input)

summary(model, input_data=[grid_input, cond_input])

Layer (type:depth-idx)                   Output Shape              Param #
MultiHeadUNET                            [1, 2, 12, 61, 61]        --
├─Encoder: 1-1                           [1, 64, 16, 64, 64]       --
│    └─FiLMConvBlock: 2-1                [1, 64, 16, 64, 64]       --
│    │    └─Conv3d: 3-1                  [1, 64, 16, 64, 64]       3,520
│    │    └─BatchNorm3d: 3-2             [1, 64, 16, 64, 64]       128
│    │    └─FiLM: 3-3                    [1, 64, 1, 1, 1]          17,536
│    │    └─GELU: 3-4                    [1, 64, 16, 64, 64]       --
│    └─FiLMConvBlock: 2-2                [1, 64, 16, 64, 64]       --
│    │    └─Conv3d: 3-5                  [1, 64, 16, 64, 64]       110,656
│    │    └─BatchNorm3d: 3-6             [1, 64, 16, 64, 64]       128
│    │    └─FiLM: 3-7                    [1, 64, 1, 1, 1]          17,536
│    │    └─GELU: 3-8                    [1, 64, 16, 64, 64]       --
│    └─MaxPool3d: 2-3                    [1, 64, 8, 32, 32]        

In [ ]:
class IndexedDataset(UHSDataset): #Added this to avoid having to rewrite the UHSDataset class
    def __init__(self, base):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        return (*self.base[idx], idx)

In [ ]:
!pip install denseweight

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.8/725.8 kB 48.0 MB/s eta 0:00:00


In [ ]:
from denseweight import DenseWeight

with_target_list = []
inj_target_list = []
inj_rate = []
with_rate = []


for i in range(len(train_dataset)):
  in_grid, in_scal, out_grid, inj_target, with_target, inpl_target = train_dataset[i]
  with_target_list.append((with_target).squeeze().item())
  inj_target_list.append((inj_target).squeeze().item())
  inj_rate.append(in_scal[0])
  with_rate.append(in_scal[1])


with_active = np.array(with_target_list)
inj_active = np.array(inj_target_list)


with_val_targets = []
inj_val_targets = []

for i in range(len(val_dataset)):
  in_grid, in_scal, out_grid, inj_target, with_target, inpl_target = val_dataset[i]
  with_val_targets.append((with_target).squeeze().item())
  inj_val_targets.append((inj_target).squeeze().item())

with_val_targets = np.array(with_val_targets)
inj_val_targets = np.array(inj_val_targets)

train_active = with_active > 0
dw = DenseWeight(alpha=1.0)
active_train_weights = dw.fit(with_active[train_active])

weights = np.zeros_like(with_active)
weights[train_active] = active_train_weights
weights = torch.tensor(weights, dtype=torch.float32)

val_active = with_val_targets > 0
active_val_weights = dw(with_val_targets[val_active])

val_weights = np.zeros_like(with_val_targets)
val_weights[val_active] = active_val_weights
val_weights = torch.tensor(val_weights, dtype=torch.float32)

######################################

train_inj_active = inj_active > 0
dw_inj = DenseWeight(alpha=1.0)
active_train_weights = dw_inj.fit(inj_active[train_inj_active])

weights_inj = np.zeros_like(inj_active)
weights_inj[train_inj_active] = active_train_weights
weights_inj = torch.tensor(weights_inj, dtype=torch.float32)

val_inj_active = inj_val_targets > 0
active_val_inj_weights = dw_inj(inj_val_targets[val_inj_active])

val_inj_weights = np.zeros_like(inj_val_targets)
val_inj_weights[val_inj_active] = active_val_inj_weights
val_inj_weights = torch.tensor(val_inj_weights, dtype=torch.float32)


In [ ]:
def with_loss_func(with_pred, with_target, batch_weights):
  """
  Computes the weighted mean squared error loss for the withdrawal predictions and targets.

  Parameters:
  ----------
  with_pred (torch.Tensor): The predicted values for the withdrawal output.
  with_target (torch.Tensor): The target values for the withdrawal output.
  batch_weights (torch.Tensor): The weights for each sample in the batch.

  Returns:
  -------
  torch.Tensor: The computed weighted mean squared error loss.
  """
  per_sample_loss = F.mse_loss(with_pred, with_target, reduction='none')
  denom = (batch_weights > 0).sum().clamp(min=1)
  weighted_loss = (per_sample_loss * batch_weights).sum() / denom
  return weighted_loss

def inj_loss_func(inj_pred, inj_target, batch_weights):
  """
  Computes the weighted mean squared error loss for the injection predictions and targets.

  Parameters:
  ----------
  inj_pred (torch.Tensor): The predicted values for the injection output.
  inj_target (torch.Tensor): The target values for the injection output.
  batch_weights (torch.Tensor): The weights for each sample in the batch.

  Returns:
  -------
  torch.Tensor: The computed weighted mean squared error loss.
  """
  per_sample_loss = F.mse_loss(inj_pred, inj_target, reduction='none')
  denom = (batch_weights > 0).sum().clamp(min=1)
  weighted_loss = (per_sample_loss * batch_weights).sum() / denom
  return weighted_loss

In [ ]:
def relative_l2_loss(pred, target, eps=1e-8):
  """
  Computes the relative L2 loss between the predicted and target tensors."""
  batch_size = pred.shape[0]
  pred_flat = pred.reshape(batch_size, -1)
  target_flat = target.reshape(batch_size, -1)
  num = torch.norm(target_flat - pred_flat, p=2, dim=1)
  denom = torch.norm(target_flat, p=2, dim=1) + eps
  return (num/denom).mean()

def combined_loss(grid_pred, grid_target, inj_pred, inj_target,
                   with_pred, with_target, inplace_pred, inplace_target,
                   with_batch_weights, inj_batch_weights,
                   grid_criterion, inj_criterion, with_criterion, inplace_criterion,
                   grid_weight=1.0, inj_weight=1.0, with_weight=1.0, inplace_weight=1.0):
    """
    Computes a combined loss for multiple outputs of a model.

    Parameters:
    ----------
    grid_pred : torch.Tensor
        Predicted values for the grid output.
    grid_target : torch.Tensor
        Target values for the grid output.
    inj_pred : torch.Tensor
        Predicted values for the injection output.
    inj_target : torch.Tensor
        Target values for the injection output.
    with_pred : torch.Tensor
        Predicted values for the withdrawal output.
    with_target : torch.Tensor
        Target values for the withdrawal output.
    inplace_pred : torch.Tensor
        Predicted values for the in-place output.
    inplace_target : torch.Tensor
        Target values for the in-place output.
    with_batch_weights : torch.Tensor
        Weights for the withdrawal DenseLoss.
    inj_batch_weights : torch.Tensor
        Weights for the injection DenseLoss.
    grid_criterion : callable
        Loss function for the grid output.
    inj_criterion : callable
        Loss function for the injection output.
    with_criterion : callable
        Loss function for the withdrawal output.
    inplace_criterion : callable
        Loss function for the in-place output.
    grid_weight : float, optional
        Weight for the grid loss (default: 1.0).
    inj_weight : float, optional
        Weight for the injection loss (default: 1.0).
    with_weight : float, optional
        Weight for the withdrawal loss (default: 1.0).
    inplace_weight : float, optional
        Weight for the in-place loss (default: 1.0).

    Returns:
    -------
    tuple
        A tuple containing the combined loss and individual losses.
    """
    grid_loss = grid_criterion(grid_pred, grid_target)
    inj_loss = inj_criterion(inj_pred.squeeze(-1), inj_target, inj_batch_weights)
    with_loss = with_criterion(with_pred.squeeze(-1), with_target, with_batch_weights)
    inplace_loss = inplace_criterion(inplace_pred.squeeze(-1), inplace_target)

    combined_loss_val = (grid_loss * grid_weight + inj_loss * inj_weight
                          + with_loss * with_weight + inplace_loss * inplace_weight)
    return combined_loss_val, grid_loss, inj_loss, with_loss, inplace_loss


The two loss functions above are the exact same. However, to make it easier to follow they were implemented twice.

In [ ]:
set_seed(42)
p = 0.6
lr = 5e-4
batch_size = 32
num_workers = 4
wd = 1e-6
epochs = 200

inj_criterion = inj_loss_func
with_criterion = with_loss_func
inplace_criterion = nn.MSELoss()
grid_criterion = relative_l2_loss

optimiser = torch.optim.Adam(params=model.parameters(), lr=lr, weight_decay=wd)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs, eta_min=1e-6)

train_dataset_indexed = IndexedDataset(train_dataset)   # wraps, doesn't modify, train_dataset
val_dataset_index = IndexedDataset(val_dataset)
test_dataset_index = IndexedDataset(test_dataset)

train_loader = DataLoader(train_dataset_indexed, batch_size=32, num_workers=num_workers, shuffle=True)
valid_loader = DataLoader(dataset=val_dataset_index, batch_size=batch_size, num_workers=num_workers, shuffle=False)
test_loader = DataLoader(dataset=test_dataset_index, batch_size=batch_size, num_workers=num_workers, shuffle=False)



In [ ]:
def plume_saturation_error_torch(S, S_hat, threshold=0.01):
    """
    Compute the plume saturation error between the true and predicted saturation values.
    This is from Real-time High-resolution CO2 geological storage prediction
    using nested fourier neural operators, Wen et al., 2023.

    Parameters:
    ----------
    S (torch.Tensor): The true saturation tensor.
    S_hat (torch.Tensor): The predicted saturation tensor.
    threshold (float): The minimum value of saturation to use for error calculation.

    Returns:
    -------
    torch.Tensor: The computed plume saturation error.
    """
    mask = (S > threshold) | (S_hat.abs() > threshold)
    denom = mask.sum()
    if denom == 0:
        return torch.tensor(0.0, device=S.device, dtype=S.dtype)
    numerator = (mask * (S - S_hat).abs()).sum()
    return numerator / denom

def relative_pressure_error_torch(P, P_hat):
    """
    Compute the relative pressure error between the true and predicted pressure values.
    This is from Real-time High-resolution CO2 geological storage prediction
    using nested fourier neural operators, Wen et al., 2023.

    Parameters:
    ----------
    P (torch.Tensor): The true pressure tensor.
    P_hat (torch.Tensor): The predicted pressure tensor.

    Returns:
    -------
    torch.Tensor: The computed relative pressure error.
    """
    n_T = P.shape[0]
    n_omega = P[0].numel()

    spatial_axes = tuple(range(1, P.dim()))
    P_max_per_t = torch.amax(P, dim=spatial_axes, keepdim=True)

    abs_error = (P - P_hat).abs()
    relative_error = abs_error / P_max_per_t

    return relative_error.sum() / (n_omega * n_T)


def masked_mae(true_vals, pred_vals, threshold=1e-4):
    """
    Compute the mean absolute error between the true and predicted values, masked by a threshold.
    The threshold is to ignore values from other phases that are not relevant to the phase of interest.

    Parameters:
    ----------
    true_vals (list or np.ndarray): The true values.
    pred_vals (list or np.ndarray): The predicted values.
    threshold (float): The threshold value to mask the true values.
    Returns:
    -------
    float: The computed masked mean absolute error. Returns NaN if no values exceed the threshold
    """
    true_arr = np.array(true_vals)
    pred_arr = np.array(pred_vals)
    mask = np.abs(true_arr) > threshold
    if mask.sum() == 0:
        return float('nan')
    return np.abs(true_arr[mask] - pred_arr[mask]).mean()

In [ ]:
def train(model, optimizer, grid_criterion, inj_criterion, with_criterion, inplace_criterion,
          data_loader, with_weights, inj_weights, scalar_weight=1.0):
    model.train()
    train_loss = 0
    train_grid_loss = 0
    train_inj_loss = 0
    train_with_loss = 0
    train_inplace_loss = 0

    for input_grid, cond, spatial_target, inj_target, with_target, inplace_target, idx in data_loader:
        input_grid = input_grid.to(device)
        cond = cond.to(device)
        spatial_target = spatial_target.to(device)
        inj_target = inj_target.to(device)
        with_target = with_target.to(device)
        inplace_target = inplace_target.to(device)
        with_batch_weights = weights[idx].to(device)
        inj_batch_weights = weights_inj[idx].to(device)

        optimizer.zero_grad()

        spatial_pred, inj_pred, with_pred, inplace_pred = model(input_grid, cond)

        loss, grid_loss, inj_loss_val, with_loss_val, inplace_loss_val = combined_loss(
            spatial_pred, spatial_target, inj_pred, inj_target, with_pred, with_target,
            inplace_pred, inplace_target, with_batch_weights, inj_batch_weights,
            grid_criterion, inj_criterion, with_criterion, inplace_criterion
        )

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * input_grid.size(0)
        train_grid_loss += grid_loss.item() * input_grid.size(0)
        train_inj_loss += inj_loss_val.item() * input_grid.size(0)
        train_with_loss += with_loss_val.item() * input_grid.size(0)
        train_inplace_loss += inplace_loss_val.item() * input_grid.size(0)

    n = len(data_loader.dataset)
    train_loss = train_loss / n
    grid_loss_f = train_grid_loss / n
    inj_loss = train_inj_loss / n
    with_loss = train_with_loss / n
    inplace_loss = train_inplace_loss / n

    return train_loss, grid_loss_f, inj_loss, with_loss, inplace_loss


def valid(model, grid_criterion, inj_criterion, with_criterion, inplace_criterion,
          data_loader, val_dataset, val_weights, val_inj_weights):
    model.eval()
    valid_loss = 0
    valid_grid_loss = 0
    valid_inj_loss = 0
    valid_with_loss = 0
    valid_inplace_loss = 0

    all_sat_true, all_sat_pred = [], []
    all_press_true, all_press_pred = [], []
    all_inj_true, all_inj_pred = [], []
    all_with_true, all_with_pred = [], []
    all_inplace_true, all_inplace_pred = [], []

    with torch.no_grad():
        for input_grid, cond, spatial_target, inj_target, with_target, inplace_target, idx in data_loader:
            input_grid = input_grid.to(device)
            cond = cond.to(device)
            spatial_target = spatial_target.to(device)
            inj_target = inj_target.to(device)
            with_target = with_target.to(device)
            inplace_target = inplace_target.to(device)
            with_batch_weights = val_weights[idx].to(device)
            inj_batch_weights = val_inj_weights[idx].to(device)

            spatial_pred, inj_pred, with_pred, inplace_pred = model(input_grid, cond)

            loss, grid_loss, inj_loss_val, with_loss_val, inplace_loss_val = combined_loss(
                spatial_pred, spatial_target, inj_pred, inj_target,
                with_pred, with_target, inplace_pred, inplace_target, with_batch_weights, inj_batch_weights,
                grid_criterion, inj_criterion, with_criterion, inplace_criterion
            )
            valid_loss += loss.item() * input_grid.size(0)
            valid_grid_loss += grid_loss.item() * input_grid.size(0)
            valid_inj_loss += inj_loss_val.item() * input_grid.size(0)
            valid_with_loss += with_loss_val.item() * input_grid.size(0)
            valid_inplace_loss += inplace_loss_val.item() * input_grid.size(0)

            all_sat_true.append(spatial_target[:, 0].cpu())
            all_sat_pred.append(spatial_pred[:, 0].cpu())
            all_press_true.append(spatial_target[:, 1].cpu())
            all_press_pred.append(spatial_pred[:, 1].cpu())

            all_inj_true.append(inj_target.cpu())
            all_inj_pred.append(inj_pred.cpu())
            all_with_true.append(with_target.cpu())
            all_with_pred.append(with_pred.cpu())
            all_inplace_true.append(inplace_target.cpu())
            all_inplace_pred.append(inplace_pred.cpu())

        n = len(data_loader.dataset)
        valid_loss = valid_loss / n
        grid_loss_f = valid_grid_loss / n
        inj_loss_f = valid_inj_loss / n
        with_loss_f = valid_with_loss / n
        inplace_loss_f = valid_inplace_loss / n

        rand_int = torch.randint(0, len(val_dataset), (1,)).item()
        (valid_grid, valid_cond, val_spatial_target,
         val_inj_target, val_with_target, val_inplace_target) = val_dataset[rand_int]

        spatial_sample, inj_sample, with_sample, inplace_sample = model(
            valid_grid.unsqueeze(0).to(device),
            valid_cond.unsqueeze(0).to(device)
        )
        spatial_sample = spatial_sample.squeeze(0).cpu()

        S_true = torch.cat(all_sat_true)
        S_pred = torch.cat(all_sat_pred)
        delta_S = plume_saturation_error_torch(S_true, S_pred).item()

        P_true = torch.cat(all_press_true)
        P_pred = torch.cat(all_press_pred)
        delta_P = relative_pressure_error_torch(P_true, P_pred).item()

        inj_true_all = torch.cat(all_inj_true)
        inj_pred_all = torch.cat(all_inj_pred)
        with_true_all = torch.cat(all_with_true)
        with_pred_all = torch.cat(all_with_pred)
        inplace_true_all = torch.cat(all_inplace_true)
        inplace_pred_all = torch.cat(all_inplace_pred)

        inj_mae = (inj_true_all - inj_pred_all).abs().mean().item()
        with_mae = (with_true_all - with_pred_all).abs().mean().item()
        inplace_mae = (inplace_true_all - inplace_pred_all).abs().mean().item()

        inj_masked_mae = masked_mae(inj_true_all.numpy(), inj_pred_all.numpy())
        with_masked_mae = masked_mae(with_true_all.numpy(), with_pred_all.numpy())

        plot_dataset(val_spatial_target, spatial_sample, view='top', position=0, index=0, prediction=True)
        plot_dataset(val_spatial_target, spatial_sample, view='North-South', position=31, index=0, prediction=True)

        return (valid_loss, delta_S, delta_P, grid_loss_f,
                inj_loss_f, with_loss_f, inplace_loss_f,
                inj_mae, with_mae, inplace_mae,
                inj_masked_mae, with_masked_mae)


def test(model, data_loader):
    model.eval()
    spatial_targets, spatial_outputs = [], []
    inj_targets, inj_outputs = [], []
    with_targets, with_outputs = [], []
    inplace_targets, inplace_outputs = [], []

    with torch.no_grad():
        for input_grid, cond, spatial_target, inj_target, with_target, inplace_target in data_loader:
            input_grid = input_grid.to(device)
            cond = cond.to(device)
            spatial_target = spatial_target.to(device)
            inj_target = inj_target.to(device)
            with_target = with_target.to(device)
            inplace_target = inplace_target.to(device)

            spatial_pred, inj_pred, with_pred, inplace_pred = model(input_grid, cond)

            spatial_targets.append(spatial_target.cpu())
            spatial_outputs.append(spatial_pred.cpu())
            inj_targets.append(inj_target.cpu())
            inj_outputs.append(inj_pred.cpu())
            with_targets.append(with_target.cpu())
            with_outputs.append(with_pred.cpu())
            inplace_targets.append(inplace_target.cpu())
            inplace_outputs.append(inplace_pred.cpu())

    return (torch.cat(spatial_targets), torch.cat(spatial_outputs),
            torch.cat(inj_targets), torch.cat(inj_outputs),
            torch.cat(with_targets), torch.cat(with_outputs),
            torch.cat(inplace_targets), torch.cat(inplace_outputs))

In [ ]:
import torch
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def evaluate_metrics(model, loader, device):
    model.eval()
    names = ["H2 saturation", "Pressure", "Inj rate", "Prod rate", "Moles stored"]

    grid_true, grid_pred = [], []
    inj_true, inj_pred_list = [], []
    with_true, with_pred_list = [], []
    inplace_true, inplace_pred_list = [], []

    with torch.no_grad():
        for input_grid, cond, spatial_target, inj_target, with_target, inplace_target, idx in loader:
            input_grid, cond = input_grid.to(device), cond.to(device)
            spatial_pred, inj_pred, with_pred, inplace_pred = model(input_grid, cond)

            grid_true.append(spatial_target.cpu())
            grid_pred.append(spatial_pred.cpu())

            inj_true.append(inj_target.cpu())
            inj_pred_list.append(inj_pred.cpu())

            with_true.append(with_target.cpu())
            with_pred_list.append(with_pred.cpu())

            inplace_true.append(inplace_target.cpu())
            inplace_pred_list.append(inplace_pred.cpu())

    grid_true = torch.cat(grid_true).numpy()
    grid_pred = torch.cat(grid_pred).numpy()
    inj_true = torch.cat(inj_true).numpy()
    inj_pred_all = torch.cat(inj_pred_list).numpy()
    with_true = torch.cat(with_true).numpy()
    with_pred_all = torch.cat(with_pred_list).numpy()
    inplace_true = torch.cat(inplace_true).numpy()
    inplace_pred_all = torch.cat(inplace_pred_list).numpy()

    results = {}
    for i in range(2):
        yt, yp = grid_true[:, i].reshape(-1), grid_pred[:, i].reshape(-1)
        results[names[i]] = {"r2": r2_score(yt, yp), "mae": mean_absolute_error(yt, yp), "mse": mean_squared_error(yt, yp)}

    scalar_pairs = [
        ("Inj rate", inj_true, inj_pred_all),
        ("Prod rate", with_true, with_pred_all),
        ("Moles stored", inplace_true, inplace_pred_all),
    ]
    for name, yt, yp in scalar_pairs:
        yt, yp = yt.reshape(-1), yp.reshape(-1)
        results[name] = {"r2": r2_score(yt, yp), "mae": mean_absolute_error(yt, yp), "mse": mean_squared_error(yt, yp)}

    return results

### Note you will need to change the paths in the cell below to your own directory

In [ ]:
import json

log_path = "/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET_Loss"

liveloss = PlotLosses()

all_logs = []

for i in range(1, epochs+1):
    train_loss, train_grid_loss, train_inj_loss, train_with_loss, train_inplace_loss = train(
        model, optimiser, grid_criterion, inj_criterion, with_criterion, inplace_criterion,
        train_loader, weights, weights_inj
    )
    (valid_loss, delta_S, delta_P, valid_grid_loss,
     valid_inj_loss, valid_with_loss, valid_inplace_loss,
     inj_mae, with_mae, inplace_mae,
     inj_masked_mae, with_masked_mae) = valid(
        model, grid_criterion, inj_criterion, with_criterion, inplace_criterion,
        valid_loader, val_dataset, val_weights, val_inj_weights
    )

    logs = {}
    logs["log loss"] = train_loss
    logs["val_log loss"] = valid_loss
    logs["delta_S"] = delta_S
    logs["delta_P"] = delta_P
    logs["train_grid_loss"] = train_grid_loss
    logs["train_inj_loss"] = train_inj_loss
    logs["train_with_loss"] = train_with_loss
    logs["train_inplace_loss"] = train_inplace_loss
    logs["val_grid_loss"] = valid_grid_loss
    logs["val_inj_loss"] = valid_inj_loss
    logs["val_with_loss"] = valid_with_loss
    logs["val_inplace_loss"] = valid_inplace_loss
    logs["inj_mae"] = inj_mae
    logs["with_mae"] = with_mae
    logs["inplace_mae"] = inplace_mae
    logs["inj_masked_mae"] = float(inj_masked_mae)
    logs["with_masked_mae"] = float(with_masked_mae)
    logs["epoch"] = i

    liveloss.update({k: v for k, v in logs.items() if k != "epoch"})
    liveloss.send()

    all_logs.append(logs)

    with open(log_path, "w") as f:
        json.dump(all_logs, f)

    if i % 10 == 0:
        model_save_name = f'Final_UNET_epoch_{i}.pth'
        path = f"/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/{model_save_name}"
        torch.save(model.state_dict(), path)
        opt_save_name = f'Final_UNET_optimiser_epoch_{i}.pth'
        path = f"/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/{opt_save_name}"
        torch.save(optimiser.state_dict(), path)
        sch_save_name = f'Final_UNET_schedule_epoch_{i}.pth'
        path = f"/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/{sch_save_name}"
        torch.save(scheduler.state_dict(), path)
        with torch.no_grad():
            results = evaluate_metrics(model, valid_loader, device)
            for name, m in results.items():
                print(f"{name}: R2={m['r2']:.4f}, MAE={m['mae']:.4f}, MSE={m['mse']:.4f}")

In [ ]:
i = 'checkrun'

sch_save_name = f'Final_UNET_schedule_epoch_{i}.pth'
path = f"/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/{sch_save_name}"
torch.save(scheduler.state_dict(), path)

opt_save_name = f'Final_UNET_optimiser_epoch_{i}.pth'
path = f"/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/{opt_save_name}"
torch.save(optimiser.state_dict(), path)

model_save_name = f'Final_UNET_epoch_{i}.pth'
path = f"/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/{model_save_name}"
torch.save(model.state_dict(), path)


In [ ]:
i= 90
model_save_name = f'Final_UNET_epoch_{i}.pth'
path = f"/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/{model_save_name}"

result = model.load_state_dict(torch.load(path, map_location=device), strict=True)
print(result)


<All keys matched successfully>


### The Cell below generates all of the model predictions for the teacher-forced rollout. 

In [ ]:
true_inj = []
pred_inj = []

true_with = []
pred_with = []

true_inpl = []
pred_inpl = []

target_inj_wbhp = []
target_with_wbhp = []

target_pressure = []
pred_pressure = []

target_sat = []
pred_sat = []

min_pressure = 96.28905
max_pressure = 104.03392

for i in range(0, 18683):
    in_grid, in_scal, out_grid, inj_target, with_target, inpl_target = train_dataset[i]
    with torch.no_grad():
        target_wbhp_q = in_scal[2]
        target_inj_wbhp.append(target_wbhp_q)

        target_wbhp_a = in_scal[3]
        target_with_wbhp.append(target_wbhp_a)

        input_grid = in_grid.unsqueeze(0).to(device)
        input_scalars = in_scal.unsqueeze(0).to(device)
        target_grid = out_grid.unsqueeze(0).to(device)

        inj_target_b = inj_target.unsqueeze(0).to(device) if inj_target.dim() == 0 else inj_target.to(device)
        with_target_b = with_target.unsqueeze(0).to(device) if with_target.dim() == 0 else with_target.to(device)
        inpl_target_b = inpl_target.unsqueeze(0).to(device) if inpl_target.dim() == 0 else inpl_target.to(device)

        spatial_pred, inj_pred, with_pred, inplace_pred = model(input_grid, input_scalars)

        pred_inj_val = inj_pred.squeeze().item()
        true_inj_val = inj_target_b.squeeze().item()

        pred_wi_val = with_pred.squeeze().item()
        tw = with_target_b.squeeze().item()

        pred_inpl_n = inplace_pred.squeeze().item()
        tinpl = inpl_target_b.squeeze().item()

        true_inj.append(true_inj_val)
        pred_inj.append(pred_inj_val)

        true_with.append(tw)
        pred_with.append(pred_wi_val)

        true_inpl.append(tinpl)
        pred_inpl.append(pred_inpl_n)

        target_pressure.append(
            (target_grid[0, 1, :, :, :].flatten().cpu() * (max_pressure - min_pressure)) + min_pressure
        )
        pred_pressure.append(
            (spatial_pred[0, 1, :, :, :].flatten().cpu() * (max_pressure - min_pressure)) + min_pressure
        )

        target_sat.append(target_grid[0, 0, :, :, :].flatten().cpu())
        pred_sat.append(spatial_pred[0, 0, :, :, :].flatten().cpu())


### The cell below saves all of the predictions to a dict which is then downloaded from google colab and run locally to make the plots. Note you will need to change the path.

In [ ]:
import pickle
pred_dict = {
    'True_test_inj':true_inj,
    'Pred_test_inj':pred_inj,
    'True_test_with':true_with,
    'Pred_test_with':pred_with,
    'True_test_inpl':true_inpl,
    'Pred_test_inpl':pred_inpl,
    'Target_inj_wbhp':target_inj_wbhp,
    'Target_with_wbhp':target_with_wbhp

}

file_save_name = f'Final_UNET_Prediction_dict.pkl'
path = f"/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/{file_save_name}"
with open(path, 'wb') as f:
  pickle.dump(pred_dict, f)

### The cells below adds the wbhp for each site and each month to a list. These are then added to a DataFrame. The difference in WBHP between a month and the previous month is then calculated. The dataframe was then downloaded and plotted locally.

In [ ]:
import os

wbhp = []
phase = []
for index in range(len(train_dataset) + len(val_dataset) + len(test_dataset)):
  dir = (f"/content/content/drive/MyDrive/tlp25_IRP/npy_files/")
  files = sorted(os.listdir(dir))
  site_pos = index // 119
  site_idx = files[site_pos]  # 0, 1 , 2 , 3

  # value between 0 and 118 as zero_index
  month_idx = index - (site_pos * 119)
  site_dir = dir + site_idx + "/"
  scalar_arr_file = f'{site_idx}_scalars.npy'
  scalar_arr = np.load((site_dir + scalar_arr_file), allow_pickle=True)
  wbhp.append(scalar_arr[month_idx, 5])
  phase.append('withdrawal' if float(scalar_arr[month_idx, 1]) < 0 else 'injection')


In [ ]:
df = pd.DataFrame({'wbhp': wbhp, 'phase': phase})
df2 = df.loc[df['wbhp'] == 0, 'phase'] = 'idle'
df

In [ ]:
site_group = df.index // 119
df['wbhp_prev'] = df.groupby(site_group)['wbhp'].shift()
df['wbhp_diff'] = df.groupby(site_group)['wbhp'].diff()

df_clean = df.dropna(subset=['wbhp_diff'])
df_clean = df_clean[df_clean['phase'].isin(['injection', 'withdrawal'])]
df_clean = df_clean[df_clean['wbhp_prev'] != 0]

fig, ax = plt.subplots(figsize=(10, 6))

groups = [df_clean[df_clean['phase'] == p]['wbhp_diff'].values
          for p in ['injection', 'withdrawal']]

ax.boxplot(groups, labels=['injection', 'withdrawal'])
ax.set_ylabel('WBHP Difference (current - previous)')
ax.set_title('WBHP Timestep Differences by Phase')
plt.tight_layout()
plt.show()

In [ ]:
import pickle

file_save_name = f'Final_UNET_WBHP_df.pkl'
path = f"/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/{file_save_name}"
with open(path, 'wb') as f:
  pickle.dump(df, f)

In [ ]:
def plume_saturation_error_torch(S, S_hat, threshold=0.01):
    """
    Compute the plume saturation error between the true and predicted saturation values.
    This is from Real-time High-resolution CO2 geological storage prediction
    using nested fourier neural operators, Wen et al., 2023.

    Parameters:
    ----------
    S (torch.Tensor): The true saturation tensor.
    S_hat (torch.Tensor): The predicted saturation tensor.
    threshold (float): The minimum value of saturation to use for error calculation.

    Returns:
    -------
    torch.Tensor: The computed plume saturation error.
    """
    mask = (S > threshold) | (S_hat.abs() > threshold)
    denom = mask.sum()
    if denom == 0:
        return torch.tensor(0.0, device=S.device, dtype=S.dtype)
    numerator = (mask * (S - S_hat).abs()).sum()
    return numerator / denom

min_pressure = 96.28905
max_pressure = 104.03392

def relative_pressure_error_torch(P_list, P_hat_list):
    """
    Compute the pressure relative error between the true and predicted pressure values.
    This is from Real-time High-resolution CO2 geological storage prediction
    using nested fourier neural operators, Wen et al., 2023.

    Parameters:
    ----------
    P_list (list of torch.Tensor): List of true pressure tensors for each time step.
    P_hat_list (list of torch.Tensor): List of predicted pressure tensors for each time step

    Returns:
    -------
    torch.Tensor: The computed relative pressure error.
    """
    P = torch.stack(P_list)       # shape [n_T, n_omega]
    P_hat = torch.stack(P_hat_list)

    n_T, n_omega = P.shape
    spatial_axes = tuple(range(1, P.dim()))  # (1,) since already flattened

    P_max_per_t = P.amax(dim=spatial_axes, keepdim=True)

    abs_error = (P - P_hat).abs()
    relative_error = abs_error / P_max_per_t

    return relative_error.sum() / (n_omega * n_T)

In [ ]:
relative_pressure_error_torch(target_pressure, pred_pressure)

In [ ]:
tsat_arr = torch.concatenate([torch.from_numpy(np.asarray(t)) for t in target_sat]).ravel()
psat_arr = torch.concatenate([torch.from_numpy(np.asarray(p)) for p in pred_sat]).ravel()

plume_saturation_error_torch(tsat_arr, psat_arr)


### The cells below generate the mean error for each grid cell at the final month of each site. The files were then downloaded and created in petrel. These cells were generated with the help of Anthropic's Claude Sonnet 5. The code has been fully checked and validated.

In [ ]:
num_sites = len(test_dataset) // 119  # 157
nk, ni, nj = 12, 61, 61



######################################

signed_errors = np.zeros((num_sites, nk, ni, nj), dtype=np.float32)

with torch.no_grad():
    for s in range(num_sites):
        final_idx = s * 119 + 118

        in_grid, in_scal, out_grid, inj_target, with_target, inpl_target = test_dataset[final_idx]

        input_grid = in_grid.unsqueeze(0).to(device)
        input_scalars = in_scal.unsqueeze(0).to(device)
        target_grid = out_grid.unsqueeze(0).to(device)

        spatial_pred, inj_pred, with_pred, inplace_pred = model(input_grid, input_scalars)

        true_sat = target_grid[0, 0, :, :, :].cpu().numpy().reshape(nk, ni, nj)
        pred_sat = spatial_pred[0, 0, :, :, :].cpu().numpy().reshape(nk, ni, nj)

        signed_errors[s] = pred_sat - true_sat  # no abs — keep sign

mean_error_field = signed_errors.mean(axis=0)  # (nk, ni, nj)

print("min:", mean_error_field.min(), "max:", mean_error_field.max(), "overall mean:", mean_error_field.mean())

export_order = mean_error_field.transpose(0, 2, 1)  # (K, J, I) -> flatten gives I fastest

with open('/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/sat_meanerror_final_tstep.GRDECL', 'w') as f:
    f.write("sat_meanerror_final_tstep\n")
    f.write("1\n")
    f.write("SATMERR\n")
    for v in export_order.flatten():
        f.write(f"{v:.6f}\n")

In [ ]:

######################################################

num_sites = len(test_dataset) // 119  # 157
nk, ni, nj = 12, 61, 61

#{'min_fcmip': 15634616.0, 'max_fcmip': 21247800.0, 'min_fcmpr': 0.0, 'max_fcmpr': 21179.091796875, 'min_fcmir': 0.0, 'max_fcmir': 25313.69921875, 'min_pressure': np.float32(96.28905), 'max_pressure': np.float32(104.03392)}


min_pressure = 96.28905
max_pressure = 104.03392

signed_errors = np.zeros((num_sites, nk, ni, nj), dtype=np.float32)

with torch.no_grad():
    for s in range(num_sites):
        final_idx = s * 119 + 118

        in_grid, in_scal, out_grid, inj_target, with_target, inpl_target = test_dataset[final_idx]

        input_grid = in_grid.unsqueeze(0).to(device)
        input_scalars = in_scal.unsqueeze(0).to(device)
        target_grid = out_grid.unsqueeze(0).to(device)

        spatial_pred, inj_pred, with_pred, inplace_pred = model(input_grid, input_scalars)

        true_press_norm = target_grid[0, 1, :, :, :].cpu().numpy().reshape(nk, ni, nj)
        pred_press_norm = spatial_pred[0, 1, :, :, :].cpu().numpy().reshape(nk, ni, nj)

        # denormalize back to bar
        true_press_bar = true_press_norm * (max_pressure - min_pressure) + min_pressure
        pred_press_bar = pred_press_norm * (max_pressure - min_pressure) + min_pressure

        signed_errors[s] = pred_press_bar - true_press_bar

mean_error_field = signed_errors.mean(axis=0)  # units: bar

print("min:", mean_error_field.min(), "max:", mean_error_field.max(), "overall mean:", mean_error_field.mean())

export_order = mean_error_field.transpose(0, 2, 1)

with open('/content/drive/My Drive/tlp25_IRP/Modelling/Final_UNET/Pressure_meanerror_final_tstep.GRDECL', 'w') as f:
    f.write("pressure_meanerror_final_tstep\n")
    f.write("1\n")
    f.write("PMEANERR\n")
    for v in export_order.flatten():
        f.write(f"{v:.6f}\n")

### The cell below was used to calculate all the autoregressive rollout predictions. It was developed with the help of Anthropic's Claude Sonnet 5. All code was validated and all ideas for the code were by myself.

In [ ]:
SAT_CH = 0
PRESSURE_CH = 1

min_pressure = stats_dict['min_pressure']
max_pressure = stats_dict['max_pressure']

def unnormalize_pressure(p_norm):
    return p_norm * (max_pressure - min_pressure) + min_pressure

def autoregressive_prediction_all_cases(model, test_dataset, n_timesteps):
    """
    Performs autoregressive predictions for all cases in the test dataset.
    The model takes the spatial grid and hydrogen stored in the previous
    timestep as input to predict the next timestep.
    The function collects true and predicted values for injection rate,
    withdrawal rate, moles stored, saturation and pressure
    across all timesteps and sites.

    Parameters:
    ----------
    model (torch.nn.Module): The trained model to use for predictions.
    test_dataset (torch.utils.data.Dataset): The test dataset containing the input and output data.
    n_timesteps (int): The number of timesteps to predict.

    Returns:
    -------
    tuple: A tuple containing lists of true and predicted values for various outputs across all timesteps and sites.
    """
    model.eval()
    num_sites = len(test_dataset) // n_timesteps
    print(num_sites) # Remeber this should be 157 sites

    true_rate = [[] for _ in range(n_timesteps)]
    pred_rate = [[] for _ in range(n_timesteps)]
    autoregressive_rate = [[] for _ in range(n_timesteps)]

    true_inj = [[] for _ in range(n_timesteps)]
    pred_inj = [[] for _ in range(n_timesteps)]
    ar_inj = [[] for _ in range(n_timesteps)]

    true_with = [[] for _ in range(n_timesteps)]
    pred_with = [[] for _ in range(n_timesteps)]
    ar_with = [[] for _ in range(n_timesteps)]

    true_inplace = [[] for _ in range(n_timesteps)]
    pred_inplace = [[] for _ in range(n_timesteps)]
    ar_inplace_trace = [[] for _ in range(n_timesteps)]

    true_sat = [[] for _ in range(n_timesteps)]
    pred_sat = [[] for _ in range(n_timesteps)]
    ar_sat = [[] for _ in range(n_timesteps)]

    true_pressure_out = [[] for _ in range(n_timesteps)]
    pred_pressure_out = [[] for _ in range(n_timesteps)]
    ar_pressure_out = [[] for _ in range(n_timesteps)]

    pred_grid_mse = [[] for _ in range(n_timesteps)]
    auto_grid_mse = [[] for _ in range(n_timesteps)]

    with torch.no_grad():
        for c in range(num_sites):
            ar_grid = None
            ar_inplace = None

            for t in range(n_timesteps):
                i = c * n_timesteps + t

                in_grid, in_scal, out_grid, inj_target, with_target, inpl_target = test_dataset[i]
                input_grid = in_grid.unsqueeze(0).to(device)
                input_scalars = in_scal.unsqueeze(0).to(device)
                true_out_grid = out_grid.unsqueeze(0).to(device)

                input_inj_rate = in_scal[0].item()
                input_with_rate = in_scal[1].item()

                true_rate[t].append(inj_target.item() + (-1 * with_target.item()))
                true_inplace[t].append(inpl_target.item())

                spatial_pred, inj_pred, with_pred, inplace_pred = model(input_grid, input_scalars)
                pred_rate[t].append(inj_pred.item() + (-1 * with_pred.item()))
                pred_inplace[t].append(inplace_pred.item())

                if t == 0:
                    ar_grid = input_grid
                    ar_scalars = input_scalars
                else:
                    ar_scalars = input_scalars.clone()
                    ar_scalars[0, -1] = ar_inplace

                ar_spatial_pred, ar_inj_pred, ar_with_pred, ar_inplace_pred = model(ar_grid, ar_scalars)
                autoregressive_rate[t].append(ar_inj_pred.item() + (-1 * ar_with_pred.item()))
                ar_inplace_trace[t].append(ar_inplace_pred.item())

                if input_inj_rate != 0:
                    true_inj[t].append(inj_target.item())
                    pred_inj[t].append(inj_pred.item())
                    ar_inj[t].append(ar_inj_pred.item())

                if input_with_rate != 0:
                    true_with[t].append(with_target.item())
                    pred_with[t].append(with_pred.item())
                    ar_with[t].append(ar_with_pred.item())

                pred_grid_mse[t].append(torch.mean((spatial_pred - true_out_grid) ** 2).item())

                true_sat_grid = true_out_grid[:, SAT_CH].flatten().cpu()
                pred_sat_grid = spatial_pred[:, SAT_CH].flatten().cpu()

                true_pressure_grid = unnormalize_pressure(true_out_grid[:, PRESSURE_CH]).flatten().cpu()
                pred_pressure_grid = unnormalize_pressure(spatial_pred[:, PRESSURE_CH]).flatten().cpu()

                true_sat[t].append(true_sat_grid)
                pred_sat[t].append(pred_sat_grid)
                true_pressure_out[t].append(true_pressure_grid)
                pred_pressure_out[t].append(pred_pressure_grid)

                auto_grid_mse[t].append(torch.mean((ar_spatial_pred - true_out_grid) ** 2).item())

                ar_sat_grid = ar_spatial_pred[:, SAT_CH].flatten().cpu()
                ar_pressure_grid = unnormalize_pressure(ar_spatial_pred[:, PRESSURE_CH]).flatten().cpu()

                ar_sat[t].append(ar_sat_grid)
                ar_pressure_out[t].append(ar_pressure_grid)

                if t == 1 and c == 0:
                    print("ar_grid updating correctly:", not torch.equal(ar_grid, input_grid))

                ar_grid = ar_spatial_pred
                ar_inplace = ar_inplace_pred.item()

    return (
        true_rate, pred_rate, autoregressive_rate,
        true_inj, pred_inj, ar_inj,
        true_with, pred_with, ar_with,
        true_inplace, pred_inplace, ar_inplace_trace,
        true_sat, pred_sat, ar_sat,
        true_pressure_out, pred_pressure_out, ar_pressure_out,
        pred_grid_mse, auto_grid_mse,
    )

In [ ]:
(true_rate, pred_rate, autoregressive_rate,
 true_inj, pred_inj, ar_inj,
 true_with, pred_with, ar_with,
 true_inplace, pred_inplace, ar_inplace_trace,
 true_sat, pred_sat, ar_sat,
 true_pressure_out, pred_pressure_out, ar_pressure_out,
 pred_grid_mse, auto_grid_mse) = autoregressive_prediction_all_cases(
    model, test_dataset, n_timesteps=119
)

157
ar_grid updating correctly: True


In [ ]:
import pickle

# Save the results to a pickle file for loading locally and plotting.
# The uncomment lines are due to them taking a lot of space.
# As a result they were saved initially
# However, they were not downloaded locally as full lists.
# Instead the metrics were calculated and saved in a seperate file.
# They are kept here commented out as they are then loaded in the next cell for calculation.

results = {
    'true_rate': true_rate, 'pred_rate': pred_rate, 'autoregressive_rate': autoregressive_rate,
    'true_inj': true_inj, 'pred_inj': pred_inj, 'ar_inj': ar_inj,
    'true_with': true_with, 'pred_with': pred_with, 'ar_with': ar_with,
    'true_inplace': true_inplace, 'pred_inplace': pred_inplace, 'ar_inplace_trace': ar_inplace_trace,
    #'true_sat': true_sat, 'pred_sat': pred_sat, 'ar_sat': ar_sat,
    #'true_pressure': true_pressure_out, 'pred_pressure': pred_pressure_out, 'ar_pressure': ar_pressure_out,
    #'pred_grid_mse': pred_grid_mse, 'auto_grid_mse': auto_grid_mse,
}

with open('/content/drive/MyDrive/tlp25_IRP/Modelling/Final_UNET/autoregress_vals_split_inj_with_no_grid.pkl', 'wb') as f:
    pickle.dump(results, f)



### The cell below loads the original

In [3]:
import pickle

with open('/content/drive/MyDrive/tlp25_IRP/Modelling/Final_UNET/autoregress_vals_split_inj_with.pkl', 'rb') as f:
  results = pickle.load(f)


In [ ]:
import pickle
import matplotlib.pyplot as plt
import torch
import numpy as np

n_timesteps = 119


true_sat = results['true_sat']
pred_sat = results['pred_sat']
ar_sat = results['ar_sat']

true_pressure = results['true_pressure']
pred_pressure = results['pred_pressure']
ar_pressure = results['ar_pressure']

sat_normal_errors = [] # teacher forced
sat_autoregressive_errors = []

pressure_normal_errors = [] #teacher foreced
pressure_autoregressive_errors = []

for t in range(n_timesteps):
    # --- saturation ---
    true_sat_t = torch.cat(true_sat[t]).numpy()
    pred_sat_t = torch.cat(pred_sat[t]).numpy()
    ar_sat_t = torch.cat(ar_sat[t]).numpy()

    active_mask = (true_sat_t >= 0.01) | (pred_sat_t >= 0.01) # use same boundaries as plume saturation error
    true_sat_t = true_sat_t[active_mask]
    pred_sat_t = pred_sat_t[active_mask]
    ar_sat_t = ar_sat_t[active_mask]

    if len(true_sat_t) > 1:
        with np.errstate(divide='ignore', invalid='ignore'): #just ignores runtime warnings for 0 divison
            normal_pct = np.abs((true_sat_t - pred_sat_t) / true_sat_t) * 100
            auto_pct = np.abs((true_sat_t - ar_sat_t) / true_sat_t) * 100
        normal_pct[~np.isfinite(normal_pct)] = np.nan # if true value is 0  to nan to avoid messing up the
        auto_pct[~np.isfinite(auto_pct)] = np.nan # same for autoregressive
        sat_normal_errors.append(normal_pct)
        sat_autoregressive_errors.append(auto_pct)
    else:
        sat_normal_errors.append(np.array([np.nan]))
        sat_autoregressive_errors.append(np.array([np.nan]))


    true_pressure_t = torch.cat(true_pressure[t]).numpy()
    pred_pressure_t = torch.cat(pred_pressure[t]).numpy()
    ar_pressure_t = torch.cat(ar_pressure[t]).numpy()

    active_mask = np.abs(true_pressure_t) > threshold
    true_pressure_t = true_pressure_t[active_mask]
    pred_pressure_t = pred_pressure_t[active_mask]
    ar_pressure_t = ar_pressure_t[active_mask]

    if len(true_pressure_t) > 1:
        with np.errstate(divide='ignore', invalid='ignore'):
            normal_pct = np.abs((true_pressure_t - pred_pressure_t) / true_pressure_t) * 100
            auto_pct = np.abs((true_pressure_t - ar_pressure_t) / true_pressure_t) * 100
        normal_pct[~np.isfinite(normal_pct)] = np.nan
        auto_pct[~np.isfinite(auto_pct)] = np.nan
        pressure_normal_errors.append(normal_pct)
        pressure_autoregressive_errors.append(auto_pct)
    else:
        pressure_normal_errors.append(np.array([np.nan]))
        pressure_autoregressive_errors.append(np.array([np.nan]))


def summarise(err_list):
    """
    Calculates the median, 25th percentile, and 75th percentile of a list of error arrays.

    Parameters:
    err_list (list): A list of numpy arrays containing error values.

    Returns:
    tuple: A tuple containing the median, 25th percentile, and 75th percentile arrays.
    """
    median = np.array([np.nanmedian(e) for e in err_list]) # means nan values are ignored
    q25 = np.array([np.nanpercentile(e, 25) for e in err_list]) # ''
    q75 = np.array([np.nanpercentile(e, 75) for e in err_list]) # ''
    return median, q25, q75

sat_med, sat_q25, sat_q75 = summarise(sat_normal_errors)
sat_ar_med, sat_ar_q25, sat_ar_q75 = summarise(sat_autoregressive_errors)

pres_med, pres_q25, pres_q75 = summarise(pressure_normal_errors)
pres_ar_med, pres_ar_q25, pres_ar_q75 = summarise(pressure_autoregressive_errors)


### Cell below saves the results and saves them locally for plotting.

In [5]:
ape_summary = {
    'sat_normal': {'median': sat_med, 'q25': sat_q25, 'q75': sat_q75},
    'sat_autoregressive': {'median': sat_ar_med, 'q25': sat_ar_q25, 'q75': sat_ar_q75},
    'pressure_normal': {'median': pres_med, 'q25': pres_q25, 'q75': pres_q75},
    'pressure_autoregressive': {'median': pres_ar_med, 'q25': pres_ar_q25, 'q75': pres_ar_q75},
}


path = '/content/drive/MyDrive/tlp25_IRP/Modelling/Final_UNET/sat_press_abs_percent_error.pkl'
with open(path, 'wb') as f:
    pickle.dump(ape_summary, f)